# On-Device AI 시험 대비 연습 문제 — 정답 및 풀이

각 문제마다 (1) 핵심 개념 설명, (2) 문제, (3) 답안 + 풍부한 주석이 포함되어 있습니다.

**출제 범위**
- Part 1. Pruning (문제 1-7)
- Part 2. Quantization (문제 8-13)
- Part 3. Knowledge Distillation (문제 14-16)

# Part 1. Pruning

가중치의 중요도(importance)를 판단하는 기준을 이해하고, 이를 통해 불필요한 연산을 제거하는 과정을 다루는 문제입니다.

---
## 📌 개념 1. Magnitude-based Pruning (Fine-grained)

**핵심 아이디어**: "절댓값이 작은 weight는 출력에 기여가 작다"고 가정하고, 그 중 일정 비율을 0으로 만듭니다.

**Granularity(세밀함) 계층**:
- **Fine-grained (Element-wise)**: 개별 weight 하나씩 제거 — 가장 높은 유연성, 높은 정확도
- **Vector-level / Kernel-level / Channel-level**: 벡터/커널/채널 단위 제거 — 점점 하드웨어 친화적

**공식**:
1. 제거할 개수 = `round(total_elements × sparsity)`
2. importance = `|W|` (절댓값 기준)
3. threshold = importance 중 하위에서 `num_pruned`번째 값 (`torch.kthvalue`)
4. mask = `importance > threshold`

`torch.kthvalue(x, k)`는 x에서 **k번째로 작은 값**을 반환합니다.

### 문제 1. Fine-grained Magnitude Pruning

In [ ]:
def prune_weight_fine_grained(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    # 전체 원소 중 sparsity 비율만큼 제거할 개수를 계산
    num_pruned_elements = ### round(weight.numel() * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # 각 weight의 중요도 = 절댓값 (magnitude 기준)
    importance = ### torch.abs(weight) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # 하위 num_pruned_elements번째로 작은 값이 cutoff threshold
    # flatten() 후 kthvalue를 적용해 전체 분포에서 기준값을 구함
    threshold = ### torch.kthvalue(importance.flatten(), num_pruned_elements)[0] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # threshold 초과는 살리고(1), 이하는 제거(0)
    mask = ### importance > threshold ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    weight.mul_(mask)
    return mask

---
## 📌 개념 2. Channel-level Pruning

Conv weight shape: `(out_channels, in_channels, kh, kw)`. **채널 단위**로 제거하면 해당 입력 채널 전체(모든 out_channel × kh × kw)가 한 번에 0이 됩니다. 

- Fine-grained과 달리 채널 개수(`in_channels`)를 기준으로 중요도를 산출
- 중요도는 해당 채널에 속한 모든 weight의 절댓값 합 — `dim=(0, 2, 3)`에 대해 sum
- `keepdim=True`를 유지해 `expand_as(weight)`로 원래 shape을 복원

하드웨어 측면에서 channel-level은 **실제 연산량 감소**로 직접 이어지므로 추론 가속에 유리합니다.

### 문제 2. Channel-level Pruning

In [ ]:
def prune_weight_channel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    # 입력 채널 수가 pruning 단위 — weight.shape[1] 사용
    num_channels = ### weight.shape[1] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    num_pruned_channels = ### round(num_channels * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # 각 채널별 중요도 = |W|를 (out_ch, kh, kw) 차원에 대해 모두 합산
    # keepdim=True로 shape을 유지하여 이후 expand_as로 원본 복원
    importance = ### weight.abs().sum(dim=(0, 2, 3), keepdim=True) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    threshold = torch.kthvalue(importance.flatten(), num_pruned_channels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

---
## 📌 개념 3. Kernel-level Pruning

커널 1개 = `(kh × kw)` matrix 하나를 통째로 0으로 만드는 방식.

- 커널 개수 = `out_channels × in_channels`
- 중요도는 커널 내부 `(kh, kw)`에 대해 `|W|`의 합
- `keepdim=True`로 차원을 유지한 뒤 `expand_as(weight)`로 shape 복원

Fine-grained보다 sparsity 구조가 규칙적이라 가속 가능성이 더 높습니다.

### 문제 3. Kernel-level Pruning

In [ ]:
def prune_weight_kernel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    # 전체 커널 개수 = out_channels × in_channels (커널 하나 = kh×kw 매트릭스)
    num_kernels = ### weight.shape[0] * weight.shape[1] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    num_pruned_kernels = ### round(num_kernels * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # 각 커널 내부 (kh, kw)에 대해 |W|의 합을 구해 중요도 계산
    # keepdim=True로 남겨야 이후 expand_as()로 원본 shape 복원 가능
    importance = ### weight.abs().sum(dim=(2, 3), keepdim=True) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    threshold = torch.kthvalue(importance.flatten(), num_pruned_kernels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

---
## 📌 개념 4. Global vs Layer-wise Pruning

- **Layer-wise**: 모든 레이어에 동일(또는 레이어별 지정) sparsity 적용
- **Global**: 전 레이어 weight를 모두 concat하여 **하나의 threshold**로 판정

Global 방식은 중요한 레이어는 적게 prune되고 덜 중요한 레이어는 많이 prune됩니다. 모든 weight를 1D로 `torch.cat`하여 global threshold를 계산합니다.

### 문제 4. Global Magnitude Pruning

In [ ]:
def global_prune(model, sparsity):
    parameters_to_prune = []
    for name, param in model.named_parameters():
        if param.dim() > 1:
            parameters_to_prune.append(param.view(-1))

    # 모든 레이어의 weight를 하나의 1D 텐서로 concat → 전역 비교 가능
    all_weights = ### torch.cat(parameters_to_prune) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    num_elements = all_weights.numel()
    num_zeros = ### round(num_elements * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    importance = ### torch.abs(all_weights) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # 전역 threshold: 모든 레이어를 통틀어 하위 num_zeros번째 값
    threshold = ### torch.kthvalue(importance, num_zeros)[0] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    masks = {}
    for name, param in model.named_parameters():
        if param.dim() > 1:
            masks[name] = torch.abs(param.data) > threshold
    return masks

---
## 📌 개념 5. LLM Magnitude-based Pruning

CNN과 같은 원리지만, 대상이 Transformer의 `nn.Linear`입니다.

- `lm_head`는 최종 출력층이라 정확도 보존을 위해 제외하는 것이 관례
- 각 Linear 레이어마다 독립적으로 importance = `|W|` → kthvalue → mask 생성
- `W.mul_(mask)`로 실제 weight를 0으로 만듦

### 문제 5. LLM Magnitude Pruning

In [ ]:
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        # Linear만 대상, lm_head는 보존
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            num_elements = ### W.numel() ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            num_zeros = ### round(num_elements * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            importance = ### torch.abs(W) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            threshold = ### torch.kthvalue(importance.flatten(), num_zeros)[0] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            mask = ### importance > threshold ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            W.mul_(mask)

---
## 📌 개념 6. Wanda Activation Norm 수집

Wanda는 weight만 보지 않고 **입력 activation의 크기**까지 고려해 중요도를 판단합니다:

$$ S_{ij} = |W_{ij}| \cdot \|X_j\|_2 $$

Calibration 데이터를 여러 batch로 나눠 누적할 때 **L2 norm 자체를 누적하면 틀리므로** 제곱을 누적한 뒤 최종에 sqrt를 적용합니다:
- 누적: `||x||^2 = Σ x_i^2`
- 최종: `sqrt(누적값)`
- 입력 `x.shape = (hidden, batch)` → `torch.norm(x, p=2, dim=1)` 로 hidden별 L2 계산 후 제곱

### 문제 6. Wanda용 Activation L2 Norm 누적

In [ ]:
def compute_activation_norm(x):
    x = x.type(torch.float32)
    # L2 norm^2 형태로 누적 가능 (이후 sqrt_()로 최종 L2)
    # dim=1은 batch/token 축, 결과 shape=(hidden_size,)
    activation_norm = ### torch.norm(x, p=2, dim=1) ** 2 ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    return activation_norm

---
## 📌 개념 7. Wanda Pruning

**핵심 차이**: magnitude-based는 행렬 전체에서 하나의 threshold를 사용. Wanda는 **row-wise** 동일 sparsity.

- 중요도 $S_{ij} = |W_{ij}| \cdot \|X_j\|_2$
- 각 row 별로 하위 `col × sparsity`개를 제거
- `torch.kthvalue(importance, k, dim=1)` → row별 k번째 값 반환 (shape=(row,))
- broadcasting 위해 `threshold.reshape(row, 1)`로 2D로 확장

row별 동일 sparsity 덕분에 하드웨어 친화적 N:M sparsity로 확장 가능합니다.

### 문제 7. Wanda Pruning

In [ ]:
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            row, col = W.shape
            # row당 제거할 원소 개수
            num_zeros_per_row = ### round(col * sparsity) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            # 중요도 = |W| * activation_norm  (Wanda 핵심)
            importance = ### torch.abs(W) * input_feat[n] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            # 각 row 별로 독립 threshold → (row,) 형태 반환
            threshold = ### torch.kthvalue(importance, num_zeros_per_row, dim=1)[0] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            # broadcasting 위해 (row, 1)로 변형하여 row-wise 비교
            mask = ### importance > threshold.reshape(row, 1) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            W.mul_(mask)

# Part 2. Quantization

모델을 layer 단위에서 양자화/역양자화 하는 과정을 다루는 문제입니다.

---
## 📌 개념 8. Linear Quantization 공식

FP → INT 변환 기본 공식:

$$ q = \text{round}(x / s) + z $$
$$ x \approx (q - z) \cdot s $$

- `s`: scale, `z`: zero_point
- 4단계: (1) scale로 나눔 → (2) round → (3) zero_point shift → (4) bitwidth 범위로 clamp

layer의 weight/activation을 INT로 변환하는 양자화 과정의 **가장 기본이 되는 연산**입니다.

### 문제 8. Linear Quantization (FP → INT)

In [ ]:
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8):
    # Step 1: scale로 나눠 정수 범위로 스케일
    scaled_tensor = ### fp_tensor / scale ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # Step 2: 정수로 반올림
    rounded_tensor = ### torch.round(scaled_tensor) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    rounded_tensor = rounded_tensor.to(dtype)
    # Step 3: zero_point 만큼 shift (비대칭 양자화의 경우 fp=0에 대응시키는 역할)
    shifted_tensor = ### rounded_tensor + zero_point ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # Step 4: 대상 bitwidth 범위로 자름
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    return quantized_tensor

---
## 📌 개념 9. Scale & Zero Point

FP `[fp_min, fp_max]`를 INT `[q_min, q_max]`로 매핑할 때:

$$ s = \frac{\text{fp\_max} - \text{fp\_min}}{q_{\max} - q_{\min}} $$

fp=0을 정수축의 어디로 보낼지 결정하는 값이 `zero_point`:

$$ z = \text{round}(q_{\min} - \text{fp\_min} / s) $$

주의: `q_max - q_min = 2^bitwidth - 1`

### 문제 9. Quantization Scale & Zero Point 계산

In [ ]:
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    # FP 범위를 INT 범위로 선형 매핑하는 scale
    scale = ### (fp_max - fp_min) / (quantized_max - quantized_min) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # fp=0이 정수축에서 어디인지 (비대칭 양자화)
    zero_point = ### round(quantized_min - fp_min / scale) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else:
        zero_point = round(zero_point)
    return scale, int(zero_point)

---
## 📌 개념 10. Quantized Linear Output 재스케일 (역양자화로 가는 다리)

양자화된 `input × weight` (정수 연산)의 결과는 output 양자화 범위에 맞게 **다시 스케일**해야 합니다:

$$ y_q = \text{round}\left( y_{\text{int}} \cdot \frac{s_{\text{input}} \cdot s_{\text{weight}}}{s_{\text{output}}} \right) + z_{\text{output}} $$

- `int 누적값 × (s_input × s_weight / s_output)` → output 양자화 grid에 정렬
- output zero_point 만큼 shift해 비대칭 분포 대응
- per-channel weight의 경우 `weight_scale`이 벡터이므로 broadcasting을 위해 shape 조정

### 문제 10. Quantized Linear: Output 재스케일 & Zero Point 보정

In [ ]:
def quantized_linear_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    weight_scale = weight_scale.flatten().view(1, -1)
    output = output.float()

    # Step 2: (s_input * s_weight / s_output)으로 재스케일 → output 양자화 grid에 align
    output = ### output * (input_scale * weight_scale / output_scale) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # Step 3: output zero_point 만큼 shift
    output = ### output + output_zero_point ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    return output

---
## 📌 개념 11. Quantized Conv2d Output 재스케일

Linear layer와 동일한 원리를 **Convolution layer**에도 적용합니다. 다만 weight shape이 `(oc, ic, kh, kw)`로 4D이므로, `weight_scale`도 `(1, oc, 1, 1)` shape으로 broadcasting되게 view합니다.

출력 재스케일 공식은 Linear와 동일:
- `output *= (input_scale * weight_scale / output_scale)`
- `output += output_zero_point`

이 과정이 **layer 단위 양자화/역양자화의 핵심**입니다: 정수 연산으로 처리한 뒤, float scale을 다시 곱하면서 다음 layer의 양자화 grid에 맞춥니다.

### 문제 11. Quantized Conv2d: Output 재스케일 & Zero Point 보정

In [ ]:
def quantized_conv2d_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    # weight_scale shape: (oc,) → (1, oc, 1, 1)로 broadcasting 가능하게 변형
    weight_scale = weight_scale.flatten().view(1, -1, 1, 1)
    output = output.float()

    # Step 2: output 텐서를 scale (Linear와 동일한 공식, shape만 다름)
    output = ### output * (input_scale * weight_scale / output_scale) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    # Step 3: output zero_point 만큼 shift
    output = ### output + output_zero_point ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    return output

---
## 📌 개념 12. K-means Quantization

선형 양자화와 달리 **비균등** 양자화. FP weight 값들을 K-means로 `2^bitwidth`개의 클러스터로 묶고, 각 weight를 자기 클러스터의 centroid 값으로 치환합니다.

- codebook = (centroids, labels)
- **역양자화(디코딩)** 시 `centroids[labels]`로 원래 shape 복원
- layer별로 별도 codebook을 가질 수 있어 분포에 맞춘 표현이 가능

### 문제 12. K-means Quantization

In [ ]:
def k_means_quantize(fp32_tensor, bitwidth=4, codebook=None):
    if codebook is None:
        # bitwidth 비트로 표현 가능한 클러스터 수 = 2^bitwidth
        n_clusters = ### 2 ** bitwidth ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)

    # 디코딩(역양자화): labels가 가리키는 centroid 값으로 weight 대체
    quantized_tensor = ### codebook.centroids[codebook.labels] ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))
    return codebook

---
## 📌 개념 13. Centroid 업데이트 (K-means fine-tuning)

Quantization-aware fine-tuning 중 weight가 갱신되면 centroid도 다시 계산해야 양자화 품질이 유지됩니다.

각 클러스터 k의 새 centroid = **label이 k인 weight들의 평균**. 이는 K-means의 M-step(M=Maximization)과 정확히 같은 과정입니다.

### 문제 13. K-means Codebook 업데이트

In [ ]:
def update_codebook(fp32_tensor, codebook):
    n_clusters = codebook.centroids.numel()
    fp32_tensor = fp32_tensor.view(-1)
    for k in range(n_clusters):
        # label==k인 원소들의 평균값이 새 centroid
        codebook.centroids[k] = ### torch.mean(fp32_tensor[codebook.labels == k]) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

# Part 3. Knowledge Distillation

지식증류의 원리를 이해하고, Teacher 모델의 예측값과 정답값을 활용해서 Student 모델 학습에 연결시키는 과정을 다루는 문제입니다.

---
## 📌 개념 14. Knowledge Distillation (Soft Targets)

작은 Student가 큰 Teacher의 **soft 확률 분포**를 모방하게 학습. Hard label만 쓰는 CE보다 "클래스 간 유사도" 정보를 더 얻습니다.

**Temperature T**: softmax를 부드럽게. T가 클수록 분포가 평탄해져 dark knowledge 전달.

**Loss 공식** (Hinton 2015):
$$ \mathcal{L}_{KD} = T^2 \cdot \text{KL}(P_T \| P_S) \quad , \quad P = \text{softmax}(\text{logits}/T) $$

$T^2$을 곱하는 이유: softmax 미분에서 생기는 $1/T^2$ 항을 보정하기 위함.

**최종 loss**: `soft_target_loss × w1 + CE(student, label) × w2` → Teacher 예측값과 정답값을 모두 활용.

### 문제 14. Knowledge Distillation (Soft Targets KL Loss)

In [ ]:
def kd_loss_step(teacher, student, inputs, labels, T,
                 soft_target_loss_weight, ce_loss_weight, ce_loss):
    with torch.no_grad():
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)

    # Temperature T로 softmax를 부드럽게 만들기
    soft_targets = ### nn.functional.softmax(teacher_logits / T, dim=-1) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    student_prob = ### nn.functional.softmax(student_logits / T, dim=-1) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    # KL(P_T || P_S) = Σ P_T (log P_T − log P_S) ; 배치 평균 후 T^2 스케일링
    soft_targets_loss = ### torch.sum(soft_targets * (soft_targets.log() - student_prob.log())) / student_prob.size(0) * (T ** 2) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    label_loss = ce_loss(student_logits, labels)
    # Teacher의 soft target + ground-truth label을 가중합하여 Student를 지도
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
    return loss

---
## 📌 개념 15. Cosine Similarity KD (Representation-level)

Logits가 아닌 **hidden feature 벡터**를 teacher-student 간 정렬. Student의 feature extractor 품질 향상에 초점.

- `CosineEmbeddingLoss(x, y, target=1)`일 때 loss 최소화 = cosine similarity 최대화
- 최종 loss는 (1) hidden representation 정렬 + (2) CE(정답값)의 weighted sum

### 문제 15. Cosine Loss 기반 KD

In [ ]:
def cosine_kd_step(teacher, student, inputs, labels,
                   hidden_rep_loss_weight, ce_loss_weight, ce_loss, cosine_loss):
    with torch.no_grad():
        _, teacher_hidden = teacher(inputs)
    student_logits, student_hidden = student(inputs)

    # target=1인 CosineEmbeddingLoss는 두 벡터의 cosine sim을 1에 가깝게 만듬
    hidden_rep_loss = ### cosine_loss(student_hidden, teacher_hidden, target=torch.ones(inputs.size(0)).cuda()) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

---
## 📌 개념 16. Hint-based KD (MSE on Feature Map)

Teacher와 Student의 중간 feature map을 **MSE**로 정렬. Student의 채널 수가 적으면 trainable regressor(예: 1x1 conv)로 차원을 맞춤.

- `nn.MSELoss()(student_regressed, teacher_feature)` 로 직접 feature 공간에서 거리 최소화
- CE(정답값)도 함께 사용해 classification 성능 유지

### 문제 16. MSE Feature Map 기반 KD

In [ ]:
def mse_kd_step(teacher, student, inputs, labels,
                feature_map_weight, ce_loss_weight, ce_loss, mse_loss):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)
    student_logits, regressor_feature_map = student(inputs)

    # Student regressor 출력과 teacher feature map 간 MSE
    hidden_rep_loss = ### mse_loss(regressor_feature_map, teacher_feature_map) ### ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss